In [1]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.utils import resample
from tqdm import tqdm
import pandas as pd
import time
import numpy as np

##Import data
csvfile = 'marketing_campaign_cleaned_NODUMMIES'
df = pd.read_csv(csvfile)

#Synthcity
import sys
import warnings

warnings.filterwarnings("ignore")

import synthcity.logger as log
from synthcity.plugins import Plugins
from synthcity.plugins.core.dataloader import GenericDataLoader

log.add(sink=sys.stderr, level="INFO")

#Fit GAN using ALL training data
from synthcity.plugins import Plugins

syn_model_full = Plugins().get("ctgan")

##########################FUNCTIONS#################################

def dummify_columns(df):
    # Dummify marital and educational
    dummify_marital = pd.get_dummies(df['Marital_Status'],prefix='marital')
    df = pd.concat([df, dummify_marital],axis=1)

    dummify_edu = pd.get_dummies(df['Education'],prefix='education')
    df = pd.concat([df, dummify_edu], axis=1)

    # Drop transformed cols
    df.drop(columns=['Marital_Status', 'Education'], inplace=True)
    
    return df

def train_and_evaluate(df_train, df_test):
    # Split into features and target for train and test datasets
    X_train = df_train.drop('Response', axis=1)
    y_train = df_train['Response']
    X_test = df_test.drop('Response', axis=1)
    y_test = df_test['Response']
    
    # Identify overlapping columns in train and test: note that the train set is only small (7.5p), columns may be missing
    # if we add only a little bit of synth data, there is a chance that same column is missing in synth data
    common_columns = set(X_train.columns).intersection(X_test.columns)
    
    # Keep only common columns in train and test
    X_train = X_train[common_columns]
    X_test = X_test[common_columns]
    

    # Train a Random Forest model
    model = RandomForestClassifier(n_estimators = 500, max_depth = None, 
                                   max_features = 'auto', criterion = 'gini', min_samples_split = 2)
    model.fit(X_train, y_train)

    # Make predictions
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    # Calculate metrics
    report = classification_report(y_test, y_pred, output_dict=True)
    roc_auc = roc_auc_score(y_test, y_prob)

    # Store metrics
    results = {
        'accuracy': report['accuracy'],
        'precision': report['1']['precision'],
        'recall': report['1']['recall'],
        'f1': report['1']['f1-score'],
        'auc-roc': roc_auc
    }

    return results





<stdin>:1:10: fatal error: 'omp.h' file not found
#include <omp.h>
         ^~~~~~~
1 error generated.


[KeOps] Warning : omp.h header is not in the path, disabling OpenMP.
[KeOps] Warning : Cuda libraries were not detected on the system ; using cpu only mode
2023-08-07 21:48:11,382 - Created a temporary directory at /var/folders/8w/flfvck1j6m77x3j6jf0384bw0000gn/T/tmp7r9prya0
2023-08-07 21:48:11,383 - Writing /var/folders/8w/flfvck1j6m77x3j6jf0384bw0000gn/T/tmp7r9prya0/_remote_module_non_scriptable.py


In [2]:
# OTHER extrinisc FASTER had randomstate = 4

# Define synthetic set sizes


countlen = len(df)*0.7*0.85

syn_sizes = [1, 0.5*countlen, 1*countlen,  3*countlen,  5*countlen, 
            8*countlen, 12*countlen, 18*countlen, 32*countlen, 48*countlen, 64*countlen]

n_iterations = 300  # Number of bootstrapping iterations
syn_model = Plugins().get('ctgan')

# Placeholder for the results
results = []

# Bootstrap iteration loop
for i in range(n_iterations):
    # Resample entire dataset
    start_time = time.time()

    df_resampled = resample(df, replace=True)

    # Perform train/test split
    df_train_main, df_test = train_test_split(df_resampled, test_size=0.3, 
                                         stratify=df_resampled['Response'], random_state=i)

    df_train_1, df_train_2 = train_test_split(df_train_main, test_size=0.15, random_state=i, 
                                              stratify=df_train_main['Response'])

    loader = GenericDataLoader(df_train_1, target_column='Response')
    syn_model.fit(loader,cond=df_train_1['Response'].to_frame())
    # Loop through the different synthetic set sizes
    for size in syn_sizes:
        # Generate synthetic set of the current size based on resampled training data
        #syn_set = syn_model.generate(count=size,random_state=i*125).dataframe()
        syn_set = syn_model.generate(count=size,random_state=i,cond=np.random.permutation([1]*int(round(df['Response'].mean()*size)) + [0]*int(size-round(df['Response'].mean()*size)))).dataframe()
        print(syn_set['Response'].mean() * 100)
        # Add synthetic data to resampled training data
        df_train_combined = pd.concat([df_train_2, syn_set], axis=0)

        # Dummify train and test datasets
        df_train_combined = dummify_columns(df_train_combined)
        df_test_dummified = dummify_columns(df_test.copy())  # .copy() to avoid SettingWithCopyWarning

        # Train and evaluate
        metrics = train_and_evaluate(df_train_combined, df_test_dummified)

        # Store the results with additional information
        metrics['syn_size'] = size
        metrics['iteration'] = i
        results.append(metrics)

    end_time = time.time()
    execution_time = end_time - start_time
    print(f"Time: {execution_time} seconds")
    print(f"Iteration: {i} " )
    
# Convert to DataFrame
results_exc_df_1 = pd.DataFrame(results)

 45%|█████████████████▉                      | 899/2000 [03:39<04:28,  4.09it/s]


0.0
14.351145038167939
15.255530129672007
15.247776365946633
15.200487879249886
14.98808956646022
15.035253763577463
14.939656997670973
15.141843126979968
15.084002921840758
15.082295214729772
Time: 412.89473009109497 seconds
Iteration: 0 


 27%|██████████▉                             | 549/2000 [03:00<07:56,  3.05it/s]


0.0
14.80916030534351
15.179252479023647
14.891994917407878
15.13950297301418
15.235826584087661
15.18135044146605
15.04128731738302
15.07276754876974
15.122113888271349
15.103732463139844
Time: 372.1568350791931 seconds
Iteration: 1 


 35%|█████████████▉                          | 699/2000 [03:33<06:38,  3.27it/s]


...

 35%|█████████████▉                          | 699/2000 [03:04<05:43,  3.78it/s]


0.0
15.725190839694655
15.102974828375288
15.552731893265564
15.200487879249886
15.178656503096713
15.498951915136885
15.274190133389794
15.358597527570684
15.312668720424302
15.36097944406069
Time: 372.075453042984 seconds
Iteration: 135 


 35%|█████████████▉                          | 699/2000 [03:25<06:23,  3.39it/s]


0.0
15.419847328244273
14.874141876430205
14.536213468869121
14.575392590333891
14.721295855169128
14.736708378326876
14.672877408426846
14.803610985398851
14.780703147330646


KeyboardInterrupt: 

In [ ]:
results_exc_df_1.to_clipboard()